In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
addresses_bronze_path = f"{BRONZE_PATH}/addresses"
addresses_silver_path = f"{SILVER_PATH}/addresses"

In [0]:
df_addresses_bronze = spark.read.format("delta") \
    .load(addresses_bronze_path)

In [0]:
display(df_addresses_bronze.limit(20))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("address_id").orderBy(F.col("updated_at").desc())

df_addresses_silver = df_addresses_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
df_addresses_silver.write.format("delta").mode("append").save(addresses_silver_path)

In [0]:
from delta.tables import DeltaTable

addresses_silver_table = DeltaTable.forPath(
    spark,
    addresses_silver_path
)

addresses_silver_table.alias("target") \
    .merge(
        df_addresses_silver.alias("source"),
        "source.address_id = target.address_id"
    ) \
    .whenMatchedUpdate(
        set = {
            "customer_id": "source.customer_id",
            "address_type": "source.address_type",
            "address_line1": "source.address_line1",
            "address_line2": "source.address_line2",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country",
            "postal_code": "source.postal_code",
            "updated_at": "source.updated_at"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()